# Stage 1 – Data preparation
**Owner: Anurag Ganjare** · ICT6001 Applied Project · Phishing email detection

Loads the six Kaggle source files, runs the unit tests, cleans and deduplicates the emails, makes the stratified 70/15/15 split and checks for leakage. Outputs go to `outputs/splits/`.

Runtime: CPU is fine (about 2 minutes).

In [ ]:
# Setup: connect Google Drive and point to the project folder
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/phishing_project'

import os, sys, json, time, random, pickle, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, PROJECT_DIR)
warnings.filterwarnings('ignore')

SEED = 42
DATA_DIR  = os.path.join(PROJECT_DIR, 'data')
OUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
SPLIT_DIR = os.path.join(OUT_DIR, 'splits')
MODEL_DIR = os.path.join(OUT_DIR, 'models')
FIG_DIR   = os.path.join(OUT_DIR, 'figures')
RUN_LOG   = os.path.join(OUT_DIR, 'run_log.csv')
for d in (OUT_DIR, SPLIT_DIR, MODEL_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style='whitegrid')
C_SVM, C_BERT, C_PHISH, C_LEGIT = '#3B6EA5', '#D9822B', '#C8553D', '#7FB77E'

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight'); plt.show()
    print('Saved figure:', name)
def save_summary(stage, d):
    with open(os.path.join(OUT_DIR, f'summary_{stage}.json'), 'w') as f: json.dump(d, f, indent=1, default=str)
print('Project folder:', PROJECT_DIR)

In [ ]:
QUICK_TEST = False   # True = use a 4,000-email sample to check the pipeline quickly

## 1.1 Run the unit tests (pytest)

In [ ]:
!python -m pytest -q "{PROJECT_DIR}/tests"

## 1.2 Record the exact library versions used

In [ ]:
import platform, sklearn, statsmodels, matplotlib
versions = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__,
            'scikit-learn': sklearn.__version__, 'statsmodels': statsmodels.__version__, 'matplotlib': matplotlib.__version__}
try:
    import torch, transformers
    versions.update({'torch': torch.__version__, 'transformers': transformers.__version__})
except ImportError:
    pass
with open(os.path.join(OUT_DIR, 'requirements-lock.txt'), 'w') as f:
    f.write('\n'.join(f'{k}=={v}' for k, v in versions.items() if k != 'python'))
print(json.dumps(versions, indent=1))

## 1.3 Load and profile the six source files

In [ ]:
from src.preprocessing import load_sources, prepare, stratified_split, check_leakage

raw = load_sources(DATA_DIR)
print(f'Loaded {len(raw):,} emails')
print(raw.groupby('source')['label'].agg(emails='size', phishing='sum').assign(phishing_share=lambda d: (d.phishing / d.emails).round(3)))
if QUICK_TEST:
    raw = raw.dropna(subset=['label']).groupby('label').sample(n=2000, random_state=SEED)
    print('QUICK_TEST sample:', len(raw))

In [ ]:
ps = pd.crosstab(raw['source'], raw['label'].map({1: 'Phishing', 0: 'Legitimate'}))
ax = ps.plot(kind='barh', stacked=True, figsize=(8, 4), color=[C_LEGIT, C_PHISH])
ax.set_title('Emails per source corpus and class'); ax.set_xlabel('Number of emails'); ax.set_ylabel('')
savefig('fig01_emails_per_source.png')

## 1.4 Clean and deduplicate

In [ ]:
data, steps = prepare(raw)
for k, v in steps.items(): print(f'{k:35s} {v:>7,}')
print(f"\nRemoved as duplicates: {steps['after_conflicting_labels_removed'] - steps['after_deduplication']:,}")

words = data['clean'].str.split().str.len()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(words.clip(upper=1000), bins=50, color=C_SVM, alpha=.85)
ax.axvline(200, color='black', ls='--', label='about 256 DistilBERT tokens')
ax.set_title('Email length after cleaning (capped at 1,000 words)'); ax.set_xlabel('Words'); ax.set_ylabel('Emails'); ax.legend()
savefig('fig02_email_length.png')
print('Median words:', int(words.median()), '| share over 200 words:', round(float((words > 200).mean()), 3))

## 1.5 Stratified 70/15/15 split and leakage check

In [ ]:
train, val, test = stratified_split(data, seed=SEED)
check_leakage(train, val, test)
print('Leakage check passed: no email appears in more than one split\n')
sizes = pd.DataFrame({n: {'emails': len(d), 'phishing': int(d['label'].sum()), 'phishing_share': round(d['label'].mean(), 4)}
                      for n, d in [('train', train), ('validation', val), ('test', test)]}).T
print(sizes)
coverage = pd.DataFrame({n: d['source'].value_counts() for n, d in [('train', train), ('validation', val), ('test', test)]}).fillna(0).astype(int)
print('\nEmails per source in each split:'); print(coverage)
for n, d in [('train', train), ('val', val), ('test', test)]:
    d[['source', 'label', 'text', 'clean']].to_csv(os.path.join(SPLIT_DIR, f'{n}.csv'), index=False)
print('\nSplits saved to', SPLIT_DIR)

In [ ]:
sb = pd.DataFrame({n: d['label'].map({1: 'Phishing', 0: 'Legitimate'}).value_counts() for n, d in [('Train', train), ('Validation', val), ('Test', test)]}).T
ax = sb[['Legitimate', 'Phishing']].plot(kind='bar', figsize=(7, 4), color=[C_LEGIT, C_PHISH], rot=0)
ax.set_title('Class balance in each split'); ax.set_ylabel('Number of emails'); ax.margins(y=0.12)
for cont in ax.containers: ax.bar_label(cont, fmt='{:,.0f}', fontsize=8)
savefig('fig03_split_balance.png')

## 1.6 Log the run

In [ ]:
from src.run_log import log_run
summary = {'quick_test': QUICK_TEST, 'versions': versions, 'raw_rows': len(raw), 'cleaning_steps': steps,
           'split_sizes': sizes.to_dict(orient='index'), 'source_coverage': coverage.to_dict(),
           'median_words': int(words.median()), 'share_over_200_words': round(float((words > 200).mean()), 4)}
save_summary('01_preprocess', summary)
log_run(RUN_LOG, 'preprocess', 'data', {'seed': SEED, 'split': '70/15/15 stratified', 'quick_test': QUICK_TEST}, steps, 'Anurag')
print('Stage 1 complete.')